In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# --- 1D1V BGK Low-rank solver with Chebyshev collocation and explicit time-stepping ---

# --- Parameters ---
N = 32        # Number of Chebyshev collocation points in space and velocity
T_end = 1.0   # Final time
dt = 0.01     # Time step size
Nt = int(T_end / dt)
epsilon = 0.1
nu = 1.0

# Velocity domain normalization parameters (for Maxwellian shift)
def u(t, x):  # Mean velocity, zero for simplicity
    return 0.0
def T(t, x):  # Temperature, constant for simplicity
    return 1.0

# --- Create Chebyshev points on interval [-1,1] ---
def chebyshev_points(N):
    return np.cos(np.pi * np.arange(N) / (N - 1))

x_pts = chebyshev_points(N)  # spatial collocation points
v_pts = chebyshev_points(N)  # velocity collocation points

# --- Basis functions (Chebyshev polynomials of first kind) ---
def phi(i, x):
    return np.cos(i * np.arccos(x))

def dphi(i, x):
    # derivative of Chebyshev first kind (using chain rule)
    # d/dx T_i(x) = i * U_{i-1}(x), where U are Chebyshev polynomials of second kind
    # Implement simple numerical derivative here for example:
    h = 1e-8
    return (phi(i, x + h) - phi(i, x - h)) / (2*h)

def psi(k, z):
    # Velocity basis: Hermite-like or polynomial modulated Gaussians; simplified as polynomials here
    return z**k

def dpsi(k, z):
    return k * z**(k-1) if k > 0 else np.zeros_like(z)

# --- Initialize coefficients c_{ik}(t) for low-rank approx ---
max_rank = 4
c = np.random.randn(max_rank, max_rank) * 1e-2

# --- Precompute basis evaluations at collocation points ---
phi_x = np.zeros((max_rank, N))
dphi_x = np.zeros((max_rank, N))
for i in range(max_rank):
    phi_x[i,:] = phi(i, x_pts)
    dphi_x[i,:] = dphi(i, x_pts)

psi_v = np.zeros((max_rank, N))
dpsi_v = np.zeros((max_rank, N))
for k in range(max_rank):
    psi_v[k,:] = psi(k, v_pts)
    dpsi_v[k,:] = dpsi(k, v_pts)

# --- Compute normalized velocity variables z = (v - u) / sqrt(T) ---
def z(t, x, v):
    return (v - u(t,x)) / np.sqrt(T(t,x))

# --- Evaluate f at collocation points ---
def eval_f(c, t):
    f_val = np.zeros((N,N))  # shape spatial x velocity
    for i in range(max_rank):
        for k in range(max_rank):
            # Build tensor product basis and sum weighted by coefficients
            for px in range(N):
                z_val = z(t, x_pts[px], v_pts)  # array in velocity direction
                f_val[px,:] += c[i,k] * phi_x[i, px] * psi_v[k,:] # ignoring shift by z_val for simplicity
    return f_val

# --- Calculate spatial derivative at collocation points ---
def eval_df_dx(c, t):
    dfdx = np.zeros((N,N))
    for i in range(max_rank):
        for k in range(max_rank):
            for px in range(N):
                dfdx[px,:] += c[i,k] * dphi_x[i, px] * psi_v[k,:]  # ignoring derivative w.r.t velocity shifts for simplicity
    return dfdx

# --- Calculate velocity derivative at collocation points ---
def eval_df_dv(c, t):
    dfdv = np.zeros((N,N))
    for i in range(max_rank):
        for k in range(max_rank):
            for px in range(N):
                dfdv[px,:] += c[i,k] * phi_x[i, px] * dpsi_v[k,:]
    return dfdv

# --- Maxwellian equilibrium ---
def maxwellian(t, x, v):
    T_val = T(t, x)
    u_val = u(t, x)
    return 1/np.sqrt(2*np.pi*T_val) * np.exp(-(v - u_val)**2 / (2*T_val))

# --- Main time-stepping ---
for n in range(Nt):
    t_n = n * dt
    f_n = eval_f(c, t_n)
    dfdx = eval_df_dx(c, t_n)
    
    v_grid = v_pts.reshape(1, N)  # shape (1, N) for broadcasting
    
    # Explicit Euler BGK step for f at collocation points
    M = np.zeros_like(f_n)
    for px in range(N):
        M[px,:] = maxwellian(t_n, x_pts[px], v_pts)
    
    RHS = (-v_grid * dfdx + (nu / epsilon)*(M - f_n))
    f_np1_vals = f_n + dt * RHS
    
    # --- Project back to low-rank basis by least squares ---
    # build system to solve for updated coefficients c^{n+1}
    # Using least squares fitting: minimize ||sum_{i,k} c_{i,k} φ_i(x) ψ_k(v) - f_np1_vals||^2
    
    # Construct basis matrices for fitting
    Phi = np.zeros((N**2, max_rank * max_rank))
    idx = 0
    for i in range(max_rank):
        for k in range(max_rank):
            basis_vec = np.outer(phi_x[i,:], psi_v[k,:]).flatten()
            Phi[:,idx] = basis_vec
            idx += 1
    
    b = f_np1_vals.flatten()
    
    # Solve least squares for new coefficients vector c_new_vec
    c_new_vec, _, _, _ = la.lstsq(Phi, b)
    c = c_new_vec.reshape((max_rank, max_rank))
    
    # Optional: apply truncation or other rank control here
    
    # Diagnostics or visualization every few steps
    if n % 10 == 0 or n == Nt - 1:
        print(f"Step {n+1}, time {t_n + dt:.3f}")
        plt.clf()
        plt.imshow(f_np1_vals, extent=[v_pts[0], v_pts[-1], x_pts[0], x_pts[-1]], aspect='auto')
        plt.colorbar(label='f')
        plt.title(f'Distribution function at time {t_n + dt:.3f}')
        plt.xlabel('Velocity')
        plt.ylabel('Space')
        plt.pause(0.1)

plt.show()
